# 7. Transformer Encoder

**Цель:** Собрать энкодерный блок трансформера: Multi-Head Self-Attention → Add & Norm → FFN → Add & Norm. Реализовать LayerNorm, FFN, Residual connections и стекирование блоков.

---

In [ ]:
import sys, os, logging, math
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("encoder")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
log.info("Using device: %s", device)

## 7.1 Layer Normalization

**Формула:**
$$\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

- Нормализуется по последней размерности (d_model)
- $\mu, \sigma$ — среднее и дисперсия по признакам для каждого токена
- $\gamma$ (scale) и $\beta$ (shift) — обучаемые параметры
- $\epsilon$ — малая константа для численной стабильности

**Почему LayerNorm, а не BatchNorm?**
- BatchNorm зависит от batch_size и плохо работает при переменной длине
- LayerNorm нормализует каждый токен независимо
- В трансформерах LayerNorm стабильнее при обучении

In [ ]:
log.debug("Implementing LayerNorm from scratch")

class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps
    
    def forward(self, x):
        mu = x.mean(dim=-1, keepdim=True)
        sigma = x.std(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mu) / (sigma + self.eps) + self.beta

# Сравнение с nn.LayerNorm
x = torch.randn(4, 10, 32)
our_ln = LayerNorm(32)
torch_ln = nn.LayerNorm(32)

with torch.no_grad():
    our_ln.gamma.data = torch_ln.weight.data
    our_ln.beta.data = torch_ln.bias.data
    diff = (our_ln(x) - torch_ln(x)).abs().max().item()
    print(f"Max difference from nn.LayerNorm: {diff:.2e}")
    log.info("LayerNorm implementation matches nn.LayerNorm: diff=%.2e", diff)

## 7.2 Feed-Forward Network (FFN)

**Структура:** Linear → GELU → Linear
$$
\text{FFN}(x) = W_2 \cdot \text{GELU}(W_1 \cdot x + b_1) + b_2
$$

- Внутренняя размерность: d_ff = 4 * d_model
- GELU предпочтительнее ReLU в трансформерах (гладкая нелинейность)

In [ ]:
log.debug("Implementing FeedForward network")

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        log.debug("FeedForward: d_model=%d, d_ff=%d", d_model, d_ff)
    
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

ffn = FeedForward(d_model=32)
x = torch.randn(4, 10, 32)
print(f"FFN input:  {x.shape}")
print(f"FFN output: {ffn(x).shape}")
log.info("FeedForward forward pass OK")

## 7.3 Residual Connections

**Зачем остаточные связи?**
- Позволяют градиенту "обтекать" слои (shortcut)
- Решают проблему затухания градиента в глубоких сетях
- Теоретически: каждый блок учится "поправке" к входу

**Формула:** output = LayerNorm(x + Sublayer(x))

**Pre-Norm vs Post-Norm:**
1. **Post-Norm** (оригинальный Vaswani): Sublayer → Add → Norm
2. **Pre-Norm** (современный стандарт): Norm → Sublayer → Add
- Pre-Norm стабильнее при обучении (градиенты не затухают в глубоких стеках)

## 7.4 TransformerEncoderBlock

Собираем всё вместе: Self-Attention → Add & Norm → FFN → Add & Norm

In [ ]:
log.debug("Implementing TransformerEncoderBlock")

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    return attn @ V, attn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        batch = Q.size(0)
        Q = self.W_Q(Q).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(F.softmax(scores, dim=-1))
        
        output = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch, -1, self.d_model)
        return self.W_O(output), attn

class TransformerEncoderBlock(nn.Module):
    """Один блок энкодера трансформера (Pre-Norm)."""
    
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        log.debug("TransformerEncoderBlock: d_model=%d, n_heads=%d, d_ff=%d",
                  d_model, n_heads, d_ff or 4*d_model)
    
    def forward(self, x, mask=None):
        # Pre-Norm: Norm -> Attention -> Add -> Norm -> FFN -> Add
        attn_out, attn_weights = self.attention(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        x = x + self.dropout1(attn_out)
        
        ffn_out = self.ffn(self.norm2(x))
        x = x + self.dropout2(ffn_out)
        
        return x, attn_weights

block = TransformerEncoderBlock(d_model=32, n_heads=4)
log.info("TransformerEncoderBlock created")

In [ ]:
log.debug("Testing TransformerEncoderBlock forward pass")

batch, seq_len, d_model = 4, 16, 32
x = torch.randn(batch, seq_len, d_model)

output, attn_weights = block(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention:    {attn_weights.shape}")
print(f"Input is output shape match: {x.shape == output.shape}")
print(f"Output differs from input:   {not torch.allclose(x, output, atol=1e-4)}")
log.info("Encoder block forward pass OK")

## 7.5 Стекирование блоков: TransformerEncoder

Несколько энкодерных блоков, stacked последовательно.

In [ ]:
log.debug("Implementing TransformerEncoder with stacked blocks")

class TransformerEncoder(nn.Module):
    """Стопка TransformerEncoderBlock."""
    
    def __init__(self, num_layers, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        log.debug("TransformerEncoder: %d layers, d_model=%d, n_heads=%d", num_layers, d_model, n_heads)
    
    def forward(self, x, mask=None):
        all_attentions = []
        for i, layer in enumerate(self.layers):
            log.debug("Encoder layer %d forward", i)
            x, attn = layer(x, mask)
            all_attentions.append(attn)
        return x, all_attentions

encoder = TransformerEncoder(num_layers=6, d_model=64, n_heads=8)
print(f"Encoder layers: {len(encoder.layers)}")
print(f"Total params: {sum(p.numel() for p in encoder.parameters()):,}")
log.info("TransformerEncoder created, total params=%d", sum(p.numel() for p in encoder.parameters()))

In [ ]:
log.debug("Testing full encoder forward pass")

batch, seq_len, d_model = 2, 12, 64
x = torch.randn(batch, seq_len, d_model)
output, attentions = encoder(x)

print(f"Encoder input:              {x.shape}")
print(f"Encoder output:             {output.shape}")
print(f"Number of attention maps:   {len(attentions)}")
print(f"Per-layer attention shape:  {attentions[0].shape}")
log.info("Full encoder forward pass OK")

## 7.6 Визуализация: активации после каждого компонента

Посмотрим, как меняются эмбеддинги после каждого блока.

In [ ]:
log.debug("Visualizing activations across encoder layers")

encoder_small = TransformerEncoder(num_layers=4, d_model=16, n_heads=2, dropout=0.0)
x = torch.randn(1, 8, 16)

layer_outputs = [x]
current = x
for i, layer in enumerate(encoder_small.layers):
    current, attn = layer(current)
    layer_outputs.append(current.detach())

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, ax in enumerate(axes.flat):
    if i < len(layer_outputs):
        im = ax.imshow(layer_outputs[i][0].numpy(), cmap='viridis', aspect='auto')
        ax.set_title(f'After block {i}' if i > 0 else 'Input embeddings')
        ax.set_xlabel('d_model')
        ax.set_ylabel('Token position')
        plt.colorbar(im, ax=ax, fraction=0.046)
axes.flat[-1].axis('off')
plt.suptitle('Activations Flow Through Encoder Layers')
plt.tight_layout()
plt.show()
log.info("Encoder activations visualized")

In [ ]:
print("=== Transformer Encoder complete ===")
print("Topics covered:")
print("  - Layer Normalization from scratch")
print("  - Feed-Forward Network: Linear -> GELU -> Linear")
print("  - Residual connections (Pre-Norm architecture)")
print("  - Multi-Head Self-Attention in encoder")
print("  - TransformerEncoderBlock with Add & Norm")
print("  - Stacked TransformerEncoder")
print("  - Activation flow visualization through layers")
log.info("Encoder notebook complete")